## Boltz 2 inference Demo

In [9]:
boltz_endpoint = "http://89.169.110.59:8000/"

# More information: https://docs.nvidia.com/nim/bionemo/boltz2/latest/inference.html

In [10]:
import requests
import os

if __name__ == "__main__":
    url = os.environ.get("NIM_URL", boltz_endpoint + "/v1/health/ready")
    headers = {
        "content-type": "application/json"
    }
    try:
        response = requests.get(url, headers=headers)
        print(f"NIM readiness check returned {response.status_code}")
        assert response.status_code == 200, f"Unexpected status code: {response.status_code}"
    except Exception as e:
        print(f"Health query failed: {e}")

NIM readiness check returned 200


## Basic Protein Structure Prediction

In [ ]:
import requests
import json


if __name__ == "__main__":
    sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"  # Replace with your sequence value of interest
    headers = {
    "content-type": "application/json"
    }
    data = {
    "polymers": [
        {
        "id": "A",
        "molecule_type": "protein",
        "sequence": sequence
        }
    ],
    "recycling_steps": 3,
    "sampling_steps": 50,
    "diffusion_samples": 1,
    "step_scale": 1.638,
    "output_format": "mmcif"
    }
    print("Making request...")
    response = requests.post(boltz_endpoint + "/biology/mit/boltz2/predict", headers=headers, data=json.dumps(data))
    result = response.json()
    print("Structure prediction completed")
    # Access the first predicted structure
    if result.get("structures"):
        structure = result["structures"][0]
        print(f"Structure format: {structure['format']}")
        print(f"Confidence score: {result['confidence_scores'][0]}")

## Protein-Ligand Complex Prediction

In [ ]:
import requests
import json


if __name__ == "__main__":
    sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
    headers = {
    "content-type": "application/json"
    }
    data = {
    "polymers": [
        {
        "id": "A",
        "molecule_type": "protein",
        "sequence": sequence
        }
    ],
    "ligands": [
        {
        "id": "B1",
        "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"  # Aspirin
        }
    ],
    "recycling_steps": 3,
    "sampling_steps": 50,
    "output_format": "mmcif"
    }
    print("Making request...")
    response = requests.post(boltz_endpoint + "/biology/mit/boltz2/predict", headers=headers, data=json.dumps(data))
    result = response.json()
    print("Protein-ligand complex prediction completed")

## Comprehensive Example with All Features

In [6]:
import requests
import json


if __name__ == "__main__":
    # Comprehensive example with all possible fields
    headers = {
        "content-type": "application/json"
    }
    data = {
        # Required: At least one polymer
        "polymers": [
            {
                "id": "A",                          # Chain identifier
                "molecule_type": "protein",         # DNA, RNA, or protein
                "sequence": "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN",
                "cyclic": False,                    # Whether polymer is cyclic
                "modifications": [                  # Chemical modifications
                    {
                        "ccd": "SEP",               # CCD code for modification
                        "position": 15              # 1-based position to modify
                    }
                ]
                # Note: msa field would go here for proteins with MSA data
            },
            {
                "id": "B",
                "molecule_type": "dna",
                "sequence": "ATCGATCGATCG",
                "cyclic": False
            }
        ],

        # Optional: Ligands
        "ligands": [
            {
                "id": "L1",
                "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"  # Using SMILES
            },
            {
                "id": "L2",
                "ccd": "ATP"                        # Using CCD code instead
            }
        ],

        # Optional: Constraints
        "constraints": [
            {
                "constraint_type": "pocket",
                "binder": "L1",
                "contacts": [
                    {
                        "id": "A",
                        "residue_index": 25
                    }
                ]
            }
        ],

        # Optional: Prediction parameters (all with defaults shown)
        "recycling_steps": 3,                      # 1-6, controls model iterations
        "sampling_steps": 50,                      # 10-1000, diffusion sampling steps
        "diffusion_samples": 1,                    # 1-5, number of samples to generate
        "step_scale": 1.638,                       # 0.5-5.0, controls sampling temperature
        "without_potentials": False,               # Whether to include potentials
        "output_format": "mmcif",                  # Output format (currently only mmcif)
        "concatenate_msas": False                  # Whether to concatenate MSAs
    }

    print("Making comprehensive prediction request...")
    response = requests.post(boltz_endpoint + "/biology/mit/boltz2/predict",
                           headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        result = response.json()
        print(f"Prediction completed successfully!")
        print(f"Number of structures returned: {len(result['structures'])}")
        print(f"Confidence scores: {result['confidence_scores']}")
        print(f"Runtime metrics: {result.get('metrics', {})}")

        # Access first structure
        if result['structures']:
            structure = result['structures'][0]
            print(f"Structure format: {structure['format']}")
            print(f"Structure length: {len(structure['structure'])} characters")
    else:
        print(f"Request failed with status {response.status_code}")
        print(response.text)

Making comprehensive prediction request...
Prediction completed successfully!
Number of structures returned: 1
Confidence scores: [0.5129219889640808]
Runtime metrics: {'total_time_seconds': 3.427655565999885, 'input_preparation_time_seconds': 0.04239109499758342, 'dataloader_setup_time_seconds': 0.0003153040015604347, 'model_inference_time_seconds': 3.3828542520022893, 'postprocessing_time_seconds': 0.0006286610005190596, 'affinity_prediction_time_seconds': 0.0, 'response_construction_time_seconds': 0.0}
Structure format: mmcif
Structure length: 98548 characters
